In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
table_dir = str(PROCESSED_DIR / 'AEGIS-Dx-KCD/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제
print("\n[INFO] 컬럼 제거")

columns_to_drop = [
    'ad_dig_cd', 'ad_dig_cat', 'poa_cd', 'poa_cat'
]

for file in csv_files:
    try:
        kcd_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in kcd_diagnoses_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            kcd_diagnoses_df.drop(columns = present_columns, inplace = True)

        kcd_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')

        print(f"[INFO] {file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")

    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# opd_nm, depart, ad_ward에서 신부인과, 정신건강의학과 제거

import os
import glob
import pandas as pd


# 키워드 정의
keywords = "산부인과|정신건강의학과|정신의학과"

# 처리 시작
for file in csv_files:
    try:
        kcd_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        original_len = len(kcd_diagnoses_df)

        # 각 컬럼에 대해 조건에 맞는 행 제거
        condition = pd.Series([False] * len(kcd_diagnoses_df))
        for col in ['depart', 'ad_depart_nm', 'dc_depart_nm']:
            if col in kcd_diagnoses_df.columns:
                condition = condition | kcd_diagnoses_df[col].fillna('').str.contains(keywords, regex=True)

        # 해당 행 제거
        kcd_diagnoses_df = kcd_diagnoses_df[~condition]
        removed = original_len - len(kcd_diagnoses_df)

        # 덮어쓰기 저장
        kcd_diagnoses_df.to_csv(file, index=False, encoding='utf-8')
        print(f"[INFO] {os.path.basename(file)}: {removed}건 제거 완료")

    except Exception as e:
        print(f"[ERROR] 오류 발생 - {file}: {e}")


In [ ]:
# ① pt_id 치환
print("\n[INFO] ① pt_id 치환 시작")
for file in csv_files:
    try:
        kcd_diagnoses_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in kcd_diagnoses_df.columns:
            kcd_diagnoses_df['pt_id'] = kcd_diagnoses_df['pt_id'].map(patient_id_map).fillna(kcd_diagnoses_df['pt_id'])

        print(kcd_diagnoses_df.head(3))

        kcd_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"[INFO] pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# ② hos_dt ±7일 ±30분
print("\n[INFO] ② hos_dt 노이즈 적용")
for file in csv_files:
    try:
        kcd_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt' in kcd_diagnoses_df.columns:
            kcd_diagnoses_df['hos_dt'] = pd.to_datetime(kcd_diagnoses_df['hos_dt'], errors='coerce')
            kcd_diagnoses_df['hos_dt'] = kcd_diagnoses_df['hos_dt'] + kcd_diagnoses_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(kcd_diagnoses_df.head(3))

            kcd_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
#  tx_dt ±7일 ±30분
print("\n[INFO] tx_dt 노이즈 적용")
for file in csv_files:
    try:
        kcd_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'tx_dt' in kcd_diagnoses_df.columns:
            kcd_diagnoses_df['tx_dt'] = pd.to_datetime(kcd_diagnoses_df['tx_dt'], errors='coerce')
            kcd_diagnoses_df['tx_dt'] = kcd_diagnoses_df['tx_dt'] + kcd_diagnoses_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(kcd_diagnoses_df.head(3))

            kcd_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] tx_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] tx_dt 오류: {file} → {e}")


In [ ]:
# 특이질환 kcd 추출, 제거
rare_disease_codes_df = pd.read_csv(str(RAW_DIR / '희귀질환목록.csv'), sep=',', quotechar='"', lineterminator='\n')

rare_kcd_set = set(rare_disease_codes_df['KCD코드'].dropna().unique())
print(f"[INFO] 삭제 대상 kcd 개수: {len(rare_kcd_set)}")

total_before = 0
total_modified = 0

for file in csv_files:
    try:
        kcd_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'dig_cd' and 'dig_nm' in kcd_diagnoses_df.columns:
            original_len = len(kcd_diagnoses_df)
            mask = kcd_diagnoses_df['dig_cd'].isin(rare_kcd_set)
            modified_count = mask.sum()

            kcd_diagnoses_df.loc[mask, 'dig_cd'] = 'Deidentified'
            kcd_diagnoses_df.loc[mask, 'dig_nm'] = 'Deidentified'

            total_before += original_len
            total_modified += modified_count

            kcd_diagnoses_df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"[INFO] {file} -> {modified_count}개 치환됨")
        else:
            print(f"[WARN] 'dig_cd' or 'dig_nm' 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] 오류 발생: {file} --> {e}")


# 제거 비율 계산
if total_before > 0:
    modification_ratio = (total_modified / total_before) * 100
else:
    modification_ratio = 0.0


print(f"\n[INFO] 전체 데이터 수: {total_before:,}개")
print(f"\n[INFO] 총 제거 행 수: {total_modified:,}개")
print(f"\n[INFO] 제거 비율: {modification_ratio:.2f}%")
